# FlowGuard Integral Defense Experiments

This notebook evaluates **FlowGuardIntegralDefense (C1)** against the new FlowPure-weakness attack suite. It mirrors the FlowPure deep-dive workflow, but focuses on the trajectory-integral score instead of the single `t=0` velocity score.

## What This Defense Tests

FlowPure scores one point: `||v(t=0, x)||^2`. C1 asks a stronger question: is the query low-energy along a short reverse ODE trajectory? This matters for FlowBlind-style attacks because an attacker can optimize the score at `t=0` while still leaving abnormal flow energy elsewhere on the path.

Expected behavior:
- `maze_baseline` should be easy: generated/off-manifold queries have high trajectory energy.
- `a2_projected_maze` should be harder for plain FlowPure but still visible to the integral score.
- `a3_flowblind` should fail if it only regularizes the `t=0` score.
- `adaptive_adaptive` is the main stress test: it combines denoising and velocity regularization.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "notebooks" / "flow_matching_defenses"))

from flowguard_notebook_utils import (
    ATTACKS,
    apply_nature_style,
    build_single_defense_command,
    defense_rows,
    display_defense_intro,
    display_metric_table,
    plot_defense_metrics,
    plot_score_distributions,
    plot_utility_vs_detection,
    run_command,
    show_attack_query_grid,
)

apply_nature_style()
DEFENSE = "flowguard_integral"
RUN_ROOT = project_root / "runs" / "notebook" / "flowguard_suite"
RUN_EVALUATION = False
FORCE_RERUN = False
print(f"Project root: {project_root}")
print(f"Suite root: {RUN_ROOT}")

In [ ]:
display_defense_intro(DEFENSE)

In [ ]:
command = build_single_defense_command(
    project_root,
    RUN_ROOT,
    DEFENSE,
    attack_query_budget=248,
    benign_query_budget=248,
    force=FORCE_RERUN,
)
if RUN_EVALUATION:
    run_command(command, project_root)
else:
    print("RUN_EVALUATION=False, using existing artifacts if present.")
    print(" ".join(command))

In [ ]:
rows = defense_rows(RUN_ROOT, DEFENSE)
display_metric_table(rows)
plot_defense_metrics(rows, DEFENSE)
plot_utility_vs_detection(rows, DEFENSE)

## Score Distributions

These histograms compare the benign calibration stream against each attack stream. For C1 the recorded score is mirrored through the existing `flowpure_scores` metadata key, but semantically it is the trajectory-integral score.

In [ ]:
plot_score_distributions(RUN_ROOT, DEFENSE)

## Attack Query Visualization

The images below are the actual queries saved by the attack artifacts. Clean-transfer should look like natural images, while MAZE-style attacks often show repetition or generator artifacts. C1 does not classify by visual appearance directly; the visualization helps connect attack construction with the score distributions above.

In [ ]:
for attack in ATTACKS:
    show_attack_query_grid(RUN_ROOT, DEFENSE, attack, count=6)

## Reading Your Current Results

In the current run, C1 is very strong: `a2_projected_maze`, `a3_flowblind`, `adaptive_adaptive`, and `maze_baseline` are all detected at or near 100% attack detection rate, while `a1_clean_transfer` is also high. That means this integral score is not only recovering against FlowBlind, it is also separating the clean-transfer pool from the benign reference used in this run. Treat that A1 result carefully: it may reflect a dataset shift between CIFAR-100 clean-transfer queries and the CIFAR-10 benign reference, not necessarily generic stealing behavior.